# --- 0. 라이브러리 설치 (필요시 최초 1회만 실행) ---

In [1]:
pip install imbalanced-learn scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


# --- 1. 데이터 로드 및 기본 탐색 ---

In [2]:
import pandas as pd
import numpy as np
 
df = pd.read_csv('creditcard.csv')
 
print(df.head())
print(df.info())
print(df.describe())
 
# Class 비율 확인
print("\n[원본 Class 개수]")
print(df['Class'].value_counts())
print("\n[원본 Class 비율]")
print(df['Class'].value_counts(normalize=True))
 
# 정상/사기 건수 명시적으로 확인
n_normal = (df['Class'] == 0).sum()
n_fraud = (df['Class'] == 1).sum()
print(f"\n정상 거래: {n_normal}건, 사기 거래: {n_fraud}건")

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

# --- 2. 샘플링 ---

In [3]:
# 사기 거래(Class=1)는 전부 유지, 정상 거래(Class=0)는 10,000건만 무작위 샘플링
df_fraud = df[df['Class'] == 1]
df_normal_sampled = df[df['Class'] == 0].sample(n=10000, random_state=42)
 
# 두 데이터셋 합치기
df_sampled = pd.concat([df_fraud, df_normal_sampled], axis=0).reset_index(drop=True)
 
# 샘플링 후 Class 비율 재확인
print("\n[샘플링 후 Class 개수]")
print(df_sampled['Class'].value_counts())
print("\n[샘플링 후 Class 비율]")
print(df_sampled['Class'].value_counts(normalize=True))


[샘플링 후 Class 개수]
Class
0    10000
1      492
Name: count, dtype: int64

[샘플링 후 Class 비율]
Class
0    0.953107
1    0.046893
Name: proportion, dtype: float64


# --- 3. 데이터 전처리 ---

In [4]:
from sklearn.preprocessing import StandardScaler

# 원본 Amount 표준화
scaler = StandardScaler()
df_sampled['Amount_Scaled'] = scaler.fit_transform(df_sampled[['Amount']])
 
# 원본 Amount 컬럼 제거
df_sampled = df_sampled.drop(columns=['Amount'])
 
# X, y 분리
X = df_sampled.drop(columns=['Class'])
y = df_sampled['Class']
 
print(X.head())

     Time        V1        V2        V3        V4        V5        V6  \
0   406.0 -2.312227  1.951992 -1.609851  3.997906 -0.522188 -1.426545   
1   472.0 -3.043541 -3.157307  1.088463  2.288644  1.359805 -1.064823   
2  4462.0 -2.303350  1.759247 -0.359745  2.330243 -0.821628 -0.075788   
3  6986.0 -4.397974  1.358367 -2.592844  2.679787 -1.128131 -1.706536   
4  7519.0  1.234235  3.019740 -4.304597  4.732795  3.624201 -1.357746   

         V7        V8        V9  ...       V20       V21       V22       V23  \
0 -2.537387  1.391657 -2.770089  ...  0.126911  0.517232 -0.035049 -0.465211   
1  0.325574 -0.067794 -0.270953  ...  2.102339  0.661696  0.435477  1.375966   
2  0.562320 -0.399147 -0.238253  ... -0.430022 -0.294166 -0.932391  0.172726   
3 -3.496197 -0.248778 -0.247768  ... -0.171608  0.573574  0.176968 -0.436207   
4  1.713445 -0.496358 -1.282858  ...  0.009061 -0.379068 -0.704181 -0.656805   

        V24       V25       V26       V27       V28  Amount_Scaled  
0  0.320198

# --- 4. 학습 데이터와 테스트 데이터 분할 ---

In [5]:
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
 
print("\n[Train Class 비율]")
print(y_train.value_counts(normalize=True))
print("\n[Test Class 비율]")
print(y_test.value_counts(normalize=True))


[Train Class 비율]
Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64

[Test Class 비율]
Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


# --- 5. SMOTE 적용 ---

In [6]:
# SMOTE를 적용 이유
## - 사기 거래가 정상 거래에 비해 압도적으로 적기 때문에, 모델이 정상만 예측해도 정확도(accuracy)는 높게 나오는 착시가 생김.
## - 이 경우 실제로 중요한 사기 거래를 거의 탐지하지 못하는 모델이 만들어질 위험이 큼.
## - SMOTE는 소수 클래스 샘플들 사이에 새로운 합성 샘플을 생성함으로써, 학습 데이터의 클래스 균형을 맞추고, 모델이 소수 클래스의 결정 경계를 더 잘 학습하도록 돕는다.
## - 다만, SMOTE는 반드시 학습 데이터에만 적용해야한다. (테스트 데이터에 적용하면, 실제와 다른 분포로 평가하게 되어 성능이 과대평가 된다).

from imblearn.over_sampling import SMOTE
print(f"\nSMOTE 적용 전 사기 거래 건수: {(y_train == 1).sum()}")
 
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
 
print(f"SMOTE 적용 후 사기 거래 건수: {(y_train_res == 1).sum()}")
print(f"SMOTE 적용 후 정상 거래 건수: {(y_train_res == 0).sum()}")


SMOTE 적용 전 사기 거래 건수: 394


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\min11\anaconda3\envs\ybigta\Lib\threading.py", line 1052, in _bootstrap_inner
    self.run()
  File "c:\Users\min11\anaconda3\envs\ybigta\Lib\threading.py", line 989, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\min11\anaconda3\envs\ybigta\Lib\subprocess.py", line 1597, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 4: invalid start byte


SMOTE 적용 후 사기 거래 건수: 7999
SMOTE 적용 후 정상 거래 건수: 7999


# --- 6. 모델 학습 ---

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, average_precision_score
 
# --- model_1: Logistic Regression (선형 베이스라인 모델) ---
model_1 = LogisticRegression(
    max_iter=1000,
    random_state=42
)
model_1.fit(X_train_res, y_train_res)
 
y_pred_1 = model_1.predict(X_test)
y_proba_1 = model_1.predict_proba(X_test)[:, 1]
 
print("=== model_1: Logistic Regression ===")
print(classification_report(y_test, y_pred_1, digits=4))
pr_auc_1 = average_precision_score(y_test, y_proba_1)
print(f"PR-AUC: {pr_auc_1:.4f}\n")
 
 
# --- model_2: Random Forest ---
model_2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
model_2.fit(X_train_res, y_train_res)
 
y_pred_2 = model_2.predict(X_test)
y_proba_2 = model_2.predict_proba(X_test)[:, 1]
 
print("=== model_2: Random Forest ===")
print(classification_report(y_test, y_pred_2, digits=4))
pr_auc_2 = average_precision_score(y_test, y_proba_2)
print(f"PR-AUC: {pr_auc_2:.4f}\n")
 
 
# --- model_3: Gradient Boosting ---
model_3 = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_3.fit(X_train_res, y_train_res)
 
y_pred_3 = model_3.predict(X_test)
y_proba_3 = model_3.predict_proba(X_test)[:, 1]
 
print("=== model_3: Gradient Boosting ===")
print(classification_report(y_test, y_pred_3, digits=4))
pr_auc_3 = average_precision_score(y_test, y_proba_3)
print(f"PR-AUC: {pr_auc_3:.4f}\n")

# --- 세 모델 성능 요약 비교표 ---
import pandas as pd
 
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Recall (Class 1)': [
        classification_report(y_test, y_pred_1, output_dict=True)['1']['recall'],
        classification_report(y_test, y_pred_2, output_dict=True)['1']['recall'],
        classification_report(y_test, y_pred_3, output_dict=True)['1']['recall'],
    ],
    'F1 (Class 1)': [
        classification_report(y_test, y_pred_1, output_dict=True)['1']['f1-score'],
        classification_report(y_test, y_pred_2, output_dict=True)['1']['f1-score'],
        classification_report(y_test, y_pred_3, output_dict=True)['1']['f1-score'],
    ],
    'PR-AUC': [pr_auc_1, pr_auc_2, pr_auc_3]
})
 
print("=== 모델별 성능 비교 ===")
print(results)

c:\Users\min11\anaconda3\envs\ybigta\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== model_1: Logistic Regression ===
              precision    recall  f1-score   support

           0     0.9965    0.9890    0.9927      2001
           1     0.8053    0.9286    0.8626        98

    accuracy                         0.9862      2099
   macro avg     0.9009    0.9588    0.9276      2099
weighted avg     0.9876    0.9862    0.9866      2099

PR-AUC: 0.9540

=== model_2: Random Forest ===
              precision    recall  f1-score   support

           0     0.9940    0.9970    0.9955      2001
           1     0.9348    0.8776    0.9053        98

    accuracy                         0.9914      2099
   macro avg     0.9644    0.9373    0.9504      2099
weighted avg     0.9913    0.9914    0.9913      2099

PR-AUC: 0.9533

=== model_3: Gradient Boosting ===
              precision    recall  f1-score   support

           0     0.9950    0.9920    0.9935      2001
           1     0.8462    0.8980    0.8713        98

    accuracy                         0.9876    

# --- 7. 최종 성능 평가 ---

In [8]:
# 하이퍼파라미터 튜닝 (model_2 = Random Forest 기준)

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score
 
# model_2(Random Forest)의 성능이 세 모델 중 가장 우수했으므로,
# Random Forest를 기준으로 하이퍼파라미터 튜닝을 진행합니다.
 
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}
 
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring='f1',      # 사기 탐지 문제에서는 f1 또는 average_precision이 accuracy보다 적합
    cv=3,
    n_jobs=-1
)
grid_search.fit(X_train_res, y_train_res)
 
print("최적 파라미터:", grid_search.best_params_)
 
# 원본 model_2(튜닝 전)는 그대로 보존하고, 튜닝 결과는 별도 변수에 저장(model_2_tuned)

model_2_tuned = grid_search.best_estimator_
 
y_pred_2_tuned = model_2_tuned.predict(X_test)
y_proba_2_tuned = model_2_tuned.predict_proba(X_test)[:, 1]
 
print("\n[model_2_tuned - Classification Report - Threshold 0.5]")
print(classification_report(y_test, y_pred_2_tuned, digits=4))
print(f"튜닝 후 PR-AUC: {average_precision_score(y_test, y_proba_2_tuned):.4f}")
 
# 참고: 튜닝 전 model_2 결과와 비교하고 싶다면 아래처럼 다시 출력 가능
print("\n[비교용: 튜닝 전 model_2 - Classification Report]")
print(classification_report(y_test, y_pred_2, digits=4))
print(f"튜닝 전 PR-AUC: {average_precision_score(y_test, y_proba_2):.4f}")
 
 

# Threshold 조정 (model_2_tuned 기준)

from sklearn.metrics import precision_recall_curve
import numpy as np
 
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_2_tuned)
 
# threshold별 F1-score 계산해서 가장 좋은 threshold 탐색
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])  # thresholds 길이는 precisions/recalls보다 1개 적음
best_threshold = thresholds[best_idx]
 
print(f"\n최적 Threshold: {best_threshold:.4f}")
print(f"해당 Threshold에서의 F1: {f1_scores[best_idx]:.4f}")
 
y_pred_2_adjusted = (y_proba_2_tuned >= best_threshold).astype(int)
 
print("\n[model_2_tuned Threshold 조정 후 Classification Report]")
print(classification_report(y_test, y_pred_2_adjusted, digits=4))
print(f"Threshold 조정 후 PR-AUC: {average_precision_score(y_test, y_proba_2_tuned):.4f}")
# 참고: PR-AUC는 threshold와 무관하게 확률 순위 기반이므로 threshold 조정으로 바뀌지 않음
 
 
# 최종 성능 평가 정리 (model_2_tuned 최종 결과)

target_recall = 0.80
target_f1 = 0.88
target_prauc = 0.90
 
report_dict = classification_report(y_test, y_pred_2_adjusted, output_dict=True)
 
recall_0 = report_dict['0']['recall']
recall_1 = report_dict['1']['recall']
f1_0 = report_dict['0']['f1-score']
f1_1 = report_dict['1']['f1-score']
prauc = average_precision_score(y_test, y_proba_2_tuned)
 
print("\n=== model_2_tuned 최종 성능 요약 ===")
print(f"Class 0 - Recall: {recall_0:.4f}, F1: {f1_0:.4f}")
print(f"Class 1 - Recall: {recall_1:.4f}, F1: {f1_1:.4f}")
print(f"PR-AUC: {prauc:.4f}")
 
achieved = (
    recall_0 >= target_recall and recall_1 >= target_recall and
    f1_0 >= target_f1 and f1_1 >= target_f1 and
    prauc >= target_prauc
)
print(f"\n목표 달성 여부 (Recall>=0.80, F1>=0.88, PR-AUC>=0.90, Class 0&1 모두): {achieved}")

최적 파라미터: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

[model_2_tuned - Classification Report - Threshold 0.5]
              precision    recall  f1-score   support

           0     0.9940    0.9970    0.9955      2001
           1     0.9348    0.8776    0.9053        98

    accuracy                         0.9914      2099
   macro avg     0.9644    0.9373    0.9504      2099
weighted avg     0.9913    0.9914    0.9913      2099

튜닝 후 PR-AUC: 0.9533

[비교용: 튜닝 전 model_2 - Classification Report]
              precision    recall  f1-score   support

           0     0.9940    0.9970    0.9955      2001
           1     0.9348    0.8776    0.9053        98

    accuracy                         0.9914      2099
   macro avg     0.9644    0.9373    0.9504      2099
weighted avg     0.9913    0.9914    0.9913      2099

튜닝 전 PR-AUC: 0.9533

최적 Threshold: 0.6150
해당 Threshold에서의 F1: 0.9239

[model_2_tuned Threshold 조정 후 Classification Report]
              precision    